# Off-Axis Raw Save

Saves per-clip, per-mask off-axis vectors to disk without aggregating.

**Output per clip**: `{_RAW_ROOT}/{model}/{fill}/{cls}/{clip_id}.npz`
- `v_perp` : `(N_masks, 527)` float32 — off-axis displacement vector per mask
- `ret`    : `(N_masks,)`    float32 — retention fraction (α) per mask

Checkpoint: skips a clip if its `.npz` already exists. Safe to re-run after interruption.

Storage estimate: ~19 MB per clip × 9,900 clip-conditions ≈ 191 GB total.

In [ ]:
# ── USER CONFIG ─────────────────────────────────────────────────────────────
# VP_ROOT    — output of compute_projections.py
# SAMPLES_ROOT and SIGMOID_ROOT both point to the data_extract.py output root;
# the /embs/ subdirectory is handled internally by the loader.
VP_ROOT      = "/path/to/vector_projs"     # compute_projections.py output
SAMPLES_ROOT = "/path/to/eval_data"        # data_extract.py output root
SIGMOID_ROOT = "/path/to/eval_data"        # same as SAMPLES_ROOT
OUTPUT_DIR   = "/path/to/off_axis_output"  # output: {model}/{fill}/{cls}/*.npz
#   This becomes V_PERP_ROOT in off_axis_direction.ipynb
# ─────────────────────────────────────────────────────────────────────────────
print(f"VP root     : {VP_ROOT}\n"
      f"Samples root: {SAMPLES_ROOT}\n"
      f"Sigmoid root  : {SIGMOID_ROOT}\n"
      f"Output dir  : {OUTPUT_DIR}")


In [1]:
import os
import numpy as np
import pandas as pd
os.makedirs(OUTPUT_DIR, exist_ok=True)

_VP_ROOT      = VP_ROOT
_SAMPLES_ROOT = SAMPLES_ROOT
_SIGMOID_ROOT = SIGMOID_ROOT
_RAW_ROOT     = OUTPUT_DIR

MODELS = ["panns_no_specaug", "panns_specaug_trained", "ast_wrapper"]
FILLS  = ["zero", "mean", "gaussian_noise"]

# Guitar and Music are present in the raw data but excluded from the 22 paper classes
EXCLUDE_CLASSES = {"guitar", "music"}

CLASSES = sorted(
    d for d in os.listdir(os.path.join(_SAMPLES_ROOT, MODELS[0]))
    if os.path.isdir(os.path.join(_SAMPLES_ROOT, MODELS[0], d))
    and d not in EXCLUDE_CLASSES
)

print(f"Models  : {MODELS}")
print(f"Classes : {CLASSES}  ({len(CLASSES)} total)")
print(f"Fills   : {FILLS}")
print(f"Output  : {_RAW_ROOT}")

Models  : ['panns_no_specaug', 'panns_specaug_trained', 'ast_wrapper']
Classes : ['bagpipes', 'boing', 'chicken_rooster', 'didgeridoo', 'dog', 'drum_kit', 'frog', 'frying_food', 'gunshot_gunfire', 'hair_dryer', 'harmonica', 'heart_sounds_heartbeat', 'insect', 'owl', 'rain', 'rub', 'sewing_machine', 'speech', 'thunder', 'timpani', 'train', 'whispering']  (22 total)
Fills   : ['zero', 'mean', 'gaussian_noise']
Output  : /gpfs/scratch/qp251874/workspaces/icassp_2027/off_axis


In [2]:
def load_vperp_clip(model, cls, clip_id, fill, group):
    """Compute v_perp and ret for one clip. Returns None if any file is missing."""
    npz_path     = os.path.join(_VP_ROOT,    model, cls, f"{clip_id}_{fill}.npz")
    foc_path     = os.path.join(_SIGMOID_ROOT, model, cls, "embs", f"{clip_id}_foc_{fill}.npy")
    perturb_path = os.path.join(_SIGMOID_ROOT, model, cls, "embs", f"{clip_id}_perturb_{fill}.npy")
    for p in [npz_path, foc_path, perturb_path]:
        if not os.path.exists(p):
            return None
    idx   = group["row_idx"].to_numpy(dtype=np.int32)
    ret   = np.clip(1.0 - group["occlusion_frac"].to_numpy(dtype=np.float32), 0.0, 1.0)
    v_foc = np.load(foc_path, mmap_mode="r").astype(np.float32)
    with np.load(npz_path) as npz:
        tau = npz["t"][idx].astype(np.float32)
        d   = npz["d"].astype(np.float32)
    if np.linalg.norm(d) < 1e-6:
        print(f"WARN: d near-zero for {model}/{cls}/{clip_id}/{fill} — on_axis subtraction will have no effect")
    v_m          = np.load(perturb_path, mmap_mode="r")[idx].astype(np.float32)
    displacement = v_m - v_foc[np.newaxis, :]
    on_axis      = tau[:, np.newaxis] * d[np.newaxis, :]
    return {"v_perp": displacement - on_axis, "ret": ret}


def is_valid_npz(path):
    """Return True if path exists and contains readable v_perp and ret arrays."""
    if not os.path.exists(path):
        return False
    try:
        with np.load(path) as f:
            assert "v_perp" in f and "ret" in f
            assert f["v_perp"].shape[1] == 527
        return True
    except Exception:
        return False


def check_vperp(v_perp, ret):
    """Sanity-check a loaded v_perp/ret pair. Call once before the full sweep."""
    assert v_perp.ndim == 2 and v_perp.shape[1] == 527, \
        f"Unexpected v_perp shape: {v_perp.shape}"
    assert ret.shape[0] == v_perp.shape[0], \
        f"ret length {ret.shape[0]} != v_perp rows {v_perp.shape[0]}"
    assert np.isfinite(v_perp).all(), \
        "v_perp contains non-finite values"
    assert ret.min() >= 0.0 and ret.max() <= 1.0, \
        f"ret out of [0,1]: min={ret.min():.4f} max={ret.max():.4f}"
    norms = np.linalg.norm(v_perp, axis=1)
    assert norms.mean() > 0.01, \
        f"v_perp norms suspiciously small: mean={norms.mean():.6f}"
    print(
        f"  check_vperp PASSED | shape={v_perp.shape}  "
        f"mean||v_perp||={norms.mean():.4f}  "
        f"ret=[{ret.min():.2f},{ret.max():.2f}]  "
        f"all_finite=True"
    )


print("Functions defined.")

Functions defined.


## Save loop

Iterates `model → cls → fill → clip`. Skips clips whose `.npz` already exists.
Prints one line per clip so you can watch progress and estimate remaining time.

In [3]:
_checked = False   # run check_vperp once on the first clip
n_saved  = 0
n_skip   = 0
n_corrupt = 0

for model in MODELS:
    for cls in CLASSES:

        samples_path = os.path.join(_SAMPLES_ROOT, model, cls, "samples.csv")
        if not os.path.exists(samples_path):
            print(f"SKIP (no samples.csv) {model}/{cls}")
            continue

        samples_df = pd.read_csv(
            samples_path,
            usecols=["clip_id", "fill", "occlusion_frac", "row_idx"],
            dtype={"clip_id": str, "fill": str,
                   "occlusion_frac": np.float32, "row_idx": np.int32},
        )

        for fill in FILLS:
            fill_df = samples_df[samples_df["fill"] == fill]
            if fill_df.empty:
                continue

            out_dir = os.path.join(_RAW_ROOT, model, fill, cls)
            os.makedirs(out_dir, exist_ok=True)

            for clip_id, group in fill_df.groupby("clip_id", sort=False):
                out_path = os.path.join(out_dir, f"{clip_id}.npz")

                # ── resume: skip if file exists and is readable ───────────────
                if is_valid_npz(out_path):
                    n_skip += 1
                elif os.path.exists(out_path):
                    # file exists but failed validation — corrupt partial write
                    os.remove(out_path)
                    n_corrupt += 1
                    print(f"  CORRUPT (removed, will recompute): {out_path}")

                if is_valid_npz(out_path):
                    # heartbeat every 500 total operations so resume runs aren't silent
                    if (n_saved + n_skip) % 500 == 0:
                        print(f"  saved {n_saved:>5}  skipped {n_skip:>5}  corrupt {n_corrupt}  "
                              f"last: {model}/{fill}/{cls}/{clip_id}")
                    continue

                result = load_vperp_clip(model, cls, clip_id, fill, group)
                if result is None:
                    print(f"  MISSING files: {model}/{fill}/{cls}/{clip_id}")
                    continue

                v_perp = result["v_perp"]   # (N_masks, 527)
                ret    = result["ret"]       # (N_masks,)

                if not _checked:
                    print(f"Running check_vperp on first clip ({model}/{fill}/{cls}/{clip_id}) ...")
                    check_vperp(v_perp, ret)
                    _checked = True

                np.savez(out_path, v_perp=v_perp, ret=ret)
                n_saved += 1

                if n_saved % 50 == 0:
                    print(f"  saved {n_saved:>5}  skipped {n_skip:>5}  corrupt {n_corrupt}  "
                          f"last: {model}/{fill}/{cls}/{clip_id}")

print(f"\nDone. Saved {n_saved}  skipped {n_skip}  corrupt-and-recomputed {n_corrupt}")
print(f"Output root: {_RAW_ROOT}")

Running check_vperp on first clip (panns_no_specaug/zero/bagpipes/Y0474eRAlFLY) ...
  check_vperp PASSED | shape=(9600, 527)  mean||v_perp||=42.3660  ret=[0.05,0.94]  all_finite=True
  saved    50  skipped     0  corrupt 0  last: panns_no_specaug/zero/bagpipes/YzmwoeBJpcDs
  saved   100  skipped     0  corrupt 0  last: panns_no_specaug/mean/bagpipes/YzmwoeBJpcDs
  saved   150  skipped     0  corrupt 0  last: panns_no_specaug/gaussian_noise/bagpipes/YzmwoeBJpcDs
  saved   200  skipped     0  corrupt 0  last: panns_no_specaug/zero/boing/Yz64KA0H1EUo
  saved   250  skipped     0  corrupt 0  last: panns_no_specaug/mean/boing/Yz64KA0H1EUo
  saved   300  skipped     0  corrupt 0  last: panns_no_specaug/gaussian_noise/boing/Yz64KA0H1EUo
  saved   350  skipped     0  corrupt 0  last: panns_no_specaug/zero/chicken_rooster/Yy3NZZETzVws
  saved   400  skipped     0  corrupt 0  last: panns_no_specaug/mean/chicken_rooster/Yy3NZZETzVws
  saved   450  skipped     0  corrupt 0  last: panns_no_specaug/